<a href="https://colab.research.google.com/github/LINA-LY/Language-Translation-Tool/blob/CODE/Language-Translation-Tool.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
# Language Translation Tool for Google Colab
# Multiple approaches: Free libraries + API options

# ========================================
# METHOD 1: Using googletrans (Free, No API Key)
# ========================================

# Install required libraries
!pip install googletrans==4.0.0-rc1
!pip install langdetect

import pandas as pd
from googletrans import Translator, LANGUAGES
from langdetect import detect
import time

class SimpleTranslator:
    def __init__(self):
        self.translator = Translator()
        self.supported_languages = LANGUAGES

    def detect_language(self, text):
        """Detect the language of input text"""
        try:
            detected = detect(text)
            lang_name = LANGUAGES.get(detected, "Unknown")
            return detected, lang_name
        except:
            return "unknown", "Unknown"

    def translate_text(self, text, target_lang='en', source_lang='auto'):
        """Translate text to target language"""
        try:
            # Add small delay to avoid rate limiting
            time.sleep(0.1)

            result = self.translator.translate(
                text,
                src=source_lang,
                dest=target_lang
            )

            return {
                'original': text,
                'translated': result.text,
                'source_lang': result.src,
                'target_lang': target_lang,
                'confidence': getattr(result.extra_data, 'confidence', None)
            }
        except Exception as e:
            return {
                'original': text,
                'translated': f"Translation failed: {str(e)}",
                'source_lang': source_lang,
                'target_lang': target_lang,
                'confidence': None
            }

    def show_supported_languages(self):
        """Display all supported languages"""
        print("Supported Languages:")
        print("-" * 50)
        for code, name in sorted(self.supported_languages.items()):
            print(f"{code}: {name}")

    def batch_translate(self, texts, target_lang='en', source_lang='auto'):
        """Translate multiple texts"""
        results = []
        for i, text in enumerate(texts):
            print(f"Translating {i+1}/{len(texts)}: {text[:50]}...")
            result = self.translate_text(text, target_lang, source_lang)
            results.append(result)

        return pd.DataFrame(results)

# ========================================
# METHOD 2: Using Transformers (Offline, No API)
# ========================================

!pip install transformers torch sentencepiece

from transformers import MarianMTModel, MarianTokenizer
import torch

class OfflineTranslator:
    def __init__(self):
        self.models = {}
        self.tokenizers = {}

    def load_model(self, source_lang, target_lang):
        """Load a specific translation model"""
        model_name = f"Helsinki-NLP/opus-mt-{source_lang}-{target_lang}"

        try:
            print(f"Loading model: {model_name}")
            tokenizer = MarianTokenizer.from_pretrained(model_name)
            model = MarianMTModel.from_pretrained(model_name)

            key = f"{source_lang}-{target_lang}"
            self.tokenizers[key] = tokenizer
            self.models[key] = model

            print(f"Model loaded successfully!")
            return True

        except Exception as e:
            print(f"Failed to load model {model_name}: {str(e)}")
            return False

    def translate_offline(self, text, source_lang, target_lang):
        """Translate using offline model"""
        key = f"{source_lang}-{target_lang}"

        if key not in self.models:
            if not self.load_model(source_lang, target_lang):
                return f"Model not available for {source_lang} -> {target_lang}"

        try:
            tokenizer = self.tokenizers[key]
            model = self.models[key]

            # Tokenize input
            inputs = tokenizer(text, return_tensors="pt", padding=True)

            # Generate translation
            with torch.no_grad():
                outputs = model.generate(**inputs)

            # Decode output
            translated = tokenizer.decode(outputs[0], skip_special_tokens=True)

            return translated

        except Exception as e:
            return f"Translation failed: {str(e)}"

# ========================================
# METHOD 3: Google Translate API (Requires API Key)
# ========================================

# Uncomment and use if you have Google Cloud API key
"""
!pip install google-cloud-translate

from google.cloud import translate_v2 as translate
import os

class GoogleTranslateAPI:
    def __init__(self, api_key):
        os.environ['GOOGLE_APPLICATION_CREDENTIALS'] = api_key
        self.translate_client = translate.Client()

    def translate_with_api(self, text, target_lang='en', source_lang=None):
        try:
            result = self.translate_client.translate(
                text,
                target_language=target_lang,
                source_language=source_lang
            )

            return {
                'original': text,
                'translated': result['translatedText'],
                'source_lang': result['detectedSourceLanguage'],
                'target_lang': target_lang
            }
        except Exception as e:
            return f"API translation failed: {str(e)}"
"""

# ========================================
# INTERACTIVE TRANSLATION TOOL
# ========================================

def create_translation_interface():
    """Create an interactive translation tool"""

    print("=" * 60)
    print("         LANGUAGE TRANSLATION TOOL")
    print("=" * 60)

    # Initialize translator
    translator = SimpleTranslator()

    while True:
        print("\nOptions:")
        print("1. Translate text")
        print("2. Detect language")
        print("3. Show supported languages")
        print("4. Batch translate from file")
        print("5. Exit")

        choice = input("\nEnter your choice (1-5): ").strip()

        if choice == '1':
            text = input("\nEnter text to translate: ")
            target = input("Enter target language code (e.g., 'es' for Spanish, 'fr' for French): ").strip()

            if not target:
                target = 'en'

            print("Translating...")
            result = translator.translate_text(text, target_lang=target)

            print(f"\nOriginal ({result['source_lang']}): {result['original']}")
            print(f"Translated ({result['target_lang']}): {result['translated']}")

        elif choice == '2':
            text = input("\nEnter text for language detection: ")
            lang_code, lang_name = translator.detect_language(text)
            print(f"\nDetected language: {lang_name} ({lang_code})")

        elif choice == '3':
            translator.show_supported_languages()

        elif choice == '4':
            print("\nBatch Translation:")
            texts = []
            print("Enter texts to translate (press Enter twice to finish):")

            while True:
                text = input("Text: ")
                if not text:
                    break
                texts.append(text)

            if texts:
                target = input("Target language code: ").strip() or 'en'
                print("Processing batch translation...")

                results_df = translator.batch_translate(texts, target_lang=target)
                print("\nResults:")
                print(results_df.to_string(index=False))

                # Save to CSV
                results_df.to_csv('translation_results.csv', index=False)
                print("\nResults saved to 'translation_results.csv'")

        elif choice == '5':
            print("Thank you for using the Translation Tool!")
            break

        else:
            print("Invalid choice. Please try again.")

# ========================================
# EXAMPLE USAGE
# ========================================

def demo_translation():
    """Demonstrate the translation tool"""

    print("🌍 Language Translation Tool Demo")
    print("=" * 40)

    # Initialize translator
    translator = SimpleTranslator()

    # Sample texts in different languages
    sample_texts = [
        "Hello, how are you today?",
        "Bonjour, comment allez-vous?",
        "Hola, ¿cómo estás?",
        "Guten Tag, wie geht es Ihnen?",
        "こんにちは、元気ですか？"
    ]

    print("\n1. Language Detection Demo:")
    print("-" * 30)
    for text in sample_texts:
        lang_code, lang_name = translator.detect_language(text)
        print(f"'{text}' → {lang_name} ({lang_code})")

    print("\n2. Translation Demo:")
    print("-" * 30)

    # Translate to different languages
    target_languages = ['es', 'fr', 'de', 'ja', 'zh']

    original_text = "Machine learning is transforming the world of technology."

    for target_lang in target_languages:
        result = translator.translate_text(original_text, target_lang=target_lang)
        lang_name = LANGUAGES.get(target_lang, target_lang)
        print(f"🌐 {lang_name}: {result['translated']}")

    print("\n3. Batch Translation Demo:")
    print("-" * 30)

    tech_phrases = [
        "Artificial Intelligence",
        "Machine Learning",
        "Deep Learning",
        "Neural Networks",
        "Data Science"
    ]

    results_df = translator.batch_translate(tech_phrases, target_lang='es')
    print(results_df[['original', 'translated']].to_string(index=False))

# ========================================
# RUN THE TOOL
# ========================================

if __name__ == "__main__":
    # Run the demo first
    demo_translation()

    # Then start interactive interface
    print("\n" + "="*60)
    create_translation_interface()

🌍 Language Translation Tool Demo

1. Language Detection Demo:
------------------------------
'Hello, how are you today?' → english (en)
'Bonjour, comment allez-vous?' → french (fr)
'Hola, ¿cómo estás?' → spanish (es)
'Guten Tag, wie geht es Ihnen?' → german (de)
'こんにちは、元気ですか？' → japanese (ja)

2. Translation Demo:
------------------------------
🌐 spanish: El aprendizaje automático está transformando el mundo de la tecnología.
🌐 french: L'apprentissage automatique transforme le monde de la technologie.
🌐 german: Maschinelles Lernen verändert die Welt der Technologie.
🌐 japanese: 機械学習はテクノロジーの世界を変えています。
🌐 zh: Translation failed: invalid destination language

3. Batch Translation Demo:
------------------------------
Translating 1/5: Artificial Intelligence...
Translating 2/5: Machine Learning...
Translating 3/5: Deep Learning...
Translating 4/5: Neural Networks...
Translating 5/5: Data Science...
               original              translated
Artificial Intelligence Inteligencia artificia

KeyboardInterrupt: Interrupted by user